# IndoBERT Token Classifier (Colab)

Notebook ini untuk inferensi label token pantun modern dengan skema label BIO (`B/I-ASG`, `B/I-SKT`, `B/I-DSM`, `B/I-TYP`, `O`) memakai model IndoBERT yang disimpan di Google Drive.

Sesuai dokumentasi pipeline:
- `B/I-SKT` -> Rumus A (fonemisasi alfabetis)
- `B/I-DSM` / `B/I-TYP` -> slang.csv -> g2p-id (fallback direct)
- `B/I-ASG` -> failsafe G2P asing
- `O` -> Cascaded CRF 1 (G2P kata baku)


In [ ]:
!pip -q install transformers sentencepiece accelerate g2p-id g2p-en nltk symspellpy sklearn-crfsuite joblib pyphen

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
# UBAH sesuai lokasi folder model fine-tuned Anda di Google Drive.
# Folder ini minimal berisi: config.json, pytorch_model.bin/model.safetensors,
# tokenizer.json atau vocab.txt + tokenizer config.
MODEL_DIR = '/content/drive/MyDrive/Models/indobert-pantun-ner'  # <- ganti path ini

# Lexicon slang (CSV dengan kolom: slang,formal) dari Google Drive.
SLANG_CSV_PATH = '/content/drive/MyDrive/slang.csv'  # <- ganti bila perlu

# Toggle post-processing suku kata setelah decoding BI.
# False = langsung pakai hasil decode tanpa repair tambahan.
ENABLE_POST_PROCESSING = False

# Guard agar koreksi SymSpell untuk TYP/DSM tidak terlalu agresif.
SYMSPELL_TYP_MAX_EDIT_DISTANCE = 1
SYMSPELL_TYP_MAX_LEN_DELTA = 1
SYMSPELL_TYP_REQUIRE_SAME_INITIAL = True
SYMSPELL_TYP_REQUIRE_SAME_FINAL = True

# Model CRF layer-3 (syllabifier B/I dari deret fonem hasil G2P).
CRF_BI_MODEL_PATH = '/content/drive/MyDrive/Models/crf_layer3_bi.joblib'  # <- ganti bila perlu


In [ ]:
import os
import joblib
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForTokenClassification.from_pretrained(MODEL_DIR).to(device)
model.eval()

id2label = model.config.id2label
label2id = model.config.label2id
print('Labels:', id2label)

# Preload CRF BI di bagian atas agar edit logic helper tidak memicu reload model.
crf_bi_model = None
crf_target = str(globals().get('CRF_BI_MODEL_PATH', '')).strip()
if crf_target and os.path.exists(crf_target):
    crf_bi_model = joblib.load(crf_target)
    print(f'[CRF3] Loaded model: {crf_target}')
else:
    print(f'[CRF3] Model belum dimuat (path kosong/tidak ditemukan): {crf_target}')

In [ ]:
# Setup lexicon slang CSV + SymSpell + DSM skeleton (jalankan sebelum cell helper)
import csv
import os
import re
from symspellpy import SymSpell


def _resolve_slang_csv_path() -> str:
    configured = globals().get('SLANG_CSV_PATH', '/content/drive/MyDrive/slang.csv')
    return str(configured).strip()


def _norm_lexicon_term(text: str) -> str:
    t = ''.join(ch for ch in str(text).lower().strip() if ch.isalnum())
    return re.sub(r"(.)\1{2,}", r"\1\1", t)


def _norm_formal_term(text: str) -> str:
    t = ''.join(ch for ch in str(text).lower().strip() if ch.isalpha())
    return re.sub(r"(.)\1{2,}", r"\1\1", t)


def load_slang_lexicon(csv_path: str):
    if not csv_path or not os.path.exists(csv_path):
        print(f'[SLANG] File tidak ditemukan: {csv_path}.')
        return {}, set(), set()

    slang_to_formal = {}
    slang_terms = set()
    formal_terms = set()
    with open(csv_path, 'r', encoding='utf-8-sig', newline='') as f:
        reader = csv.DictReader(f)
        if not reader.fieldnames:
            print(f'[SLANG] Header CSV tidak ditemukan: {csv_path}.')
            return {}, set(), set()
        if 'slang' not in reader.fieldnames or 'formal' not in reader.fieldnames:
            print(f"[SLANG] Header wajib 'slang,formal' tidak lengkap: {reader.fieldnames}")
            return {}, set(), set()

        for row in reader:
            slang = _norm_lexicon_term(row.get('slang', ''))
            formal = _norm_formal_term(row.get('formal', ''))
            if not slang or not formal:
                continue
            slang_to_formal[slang] = formal
            slang_terms.add(slang)
            formal_terms.add(formal)

    print(f'[SLANG] Loaded {len(slang_to_formal)} mapping term dari: {csv_path}')
    return slang_to_formal, slang_terms, formal_terms


def build_symspell_from_terms(terms: set[str], max_edit_distance: int = 2):
    if not terms:
        print('[SymSpell] Term lexicon kosong. Recover typo dinonaktifkan.')
        return None

    symspell = SymSpell(max_dictionary_edit_distance=max_edit_distance, prefix_length=7)
    loaded = 0
    for term in sorted(terms):
        if not term or not term.isalpha():
            continue
        symspell.create_dictionary_entry(term, 1)
        loaded += 1

    if loaded == 0:
        print('[SymSpell] Semua term lexicon tidak valid. Recover typo dinonaktifkan.')
        return None

    print(f'[SymSpell] Loaded {loaded} term dari formal slang.csv')
    return symspell


slang_formal_map, slang_terms, formal_terms = load_slang_lexicon(_resolve_slang_csv_path())

symspell_model = build_symspell_from_terms(formal_terms)

In [ ]:
def _clean_input_words(text: str) -> list[str]:
    words = []
    for raw in str(text).strip().split():
        cleaned = re.sub(r"^[^\w]+|[^\w]+$", "", raw)
        if cleaned:
            words.append(cleaned)
    return words


def predict_token_labels(text: str):
    """Inferensi token-level label + confidence dengan penyelarasan wordpiece -> token asli."""
    words = _clean_input_words(text)
    if not words:
        return []
    enc = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors='pt',
        truncation=True,
        padding=False
    )
    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        logits = model(**enc).logits
        probs = torch.softmax(logits, dim=-1)[0]
        pred_ids = torch.argmax(logits, dim=-1)[0].tolist()

    word_ids = tokenizer(words, is_split_into_words=True).word_ids()

    token_rows = []
    current_word = None
    for idx, widx in enumerate(word_ids):
        if widx is None:
            continue
        if widx != current_word:
            current_word = widx
            label = id2label[pred_ids[idx]]

            top2 = torch.topk(probs[idx], k=min(2, probs[idx].shape[-1]))
            confidence = float(top2.values[0].item())
            second_best = float(top2.values[1].item()) if top2.values.shape[0] > 1 else 0.0
            margin = confidence - second_best

            token_rows.append(
                {
                    'token': words[widx],
                    'label': label,
                    'confidence': confidence,
                    'margin': margin,
                }
            )
    return token_rows


LABELS_ASING = {"B-ASG", "I-ASG"}
LABELS_SINGKATAN = {"B-SKT", "I-SKT"}
LABELS_DISEMVOWEL = {"B-DSM", "I-DSM"}
LABELS_TYPO = {"B-TYP", "I-TYP"}
LABELS_VALID = {"O", *LABELS_ASING, *LABELS_SINGKATAN, *LABELS_DISEMVOWEL, *LABELS_TYPO}


def route_by_label(label: str) -> str:
    """Routing sesuai dokumentasi pipeline dan skema label BIO IndoBERT."""
    norm = label.upper().strip()
    if norm not in LABELS_VALID:
        raise ValueError(f'Label tidak dikenali: {label}')

    if norm in LABELS_SINGKATAN:
        return 'RUMUS_A_FONEMISASI_ALFABETIS'
    if norm in LABELS_DISEMVOWEL or norm in LABELS_TYPO:
        return 'SLANG_LEXICON_OR_DIRECT_G2P_ID'
    if norm in LABELS_ASING:
        return 'FAILSAFE_G2P_ASING'
    return 'CRF1_G2P_KATA_BAKU'


def _replace_label_entity(label: str, target_entity: str) -> str:
    norm = label.upper().strip()
    if norm.startswith('B-'):
        return f'B-{target_entity}'
    if norm.startswith('I-'):
        return f'I-{target_entity}'
    return target_entity


def is_likely_disemvowelling(token: str) -> bool:
    """DSM sederhana: token dianggap DSM bila ada di slang.csv."""
    mapping = globals().get('slang_formal_map') or {}
    normalized = _norm_lexicon_term(token)
    return bool(normalized and normalized in mapping)


def override_typo_to_dsm(token: str, label: str) -> str:
    norm = label.upper().strip()
    if norm in LABELS_DISEMVOWEL or norm in LABELS_TYPO:
        if is_likely_disemvowelling(token):
            return _replace_label_entity(norm, 'DSM')
        return _replace_label_entity(norm, 'TYP')
    return norm


def predict_with_routing(text: str):
    labeled = predict_token_labels(text)
    rows = []
    for item in labeled:
        tok = item['token']
        lab = item['label']
        token_meta = {
            'confidence': item.get('confidence'),
            'margin': item.get('margin'),
        }

        final_label = override_typo_to_dsm(tok, lab)
        switch = acoustic_switchboard(tok, final_label, token_meta=token_meta)
        units_now = phoneme_to_units(switch['phoneme'])

        bi_fn = globals().get('predict_bi_from_phoneme')
        bi_pattern = bi_fn(switch['phoneme']) if callable(bi_fn) else None
        bi_source = 'CRF3_BI' if bi_pattern else None

        if str(switch.get('switch_route', '')).startswith('ASG_'):
            refine_fn = globals().get('refine_asg_bi_with_pyphen')
            if callable(refine_fn):
                refined_bi, refined_source = refine_fn(tok, units_now, bi_pattern, switch.get('switch_route'))
                if refined_bi:
                    bi_pattern = refined_bi
                    bi_source = refined_source or bi_source

        rows.append(
            {
                'token': tok,
                'raw_label': lab,
                'label': final_label,
                'route': route_by_label(final_label),
                'model_confidence': token_meta['confidence'],
                'model_margin': token_meta['margin'],
                'switch_route': switch['switch_route'],
                'switch_input': switch['switch_input'],
                'phoneme': switch['phoneme'],
                'bi_pattern': bi_pattern,
                'bi_source': bi_source,
                'bc_info': switch.get('bc_info')
            }
        )
    return rows


In [ ]:
# Helper sederhana Rumus A (singkatan)
from functools import lru_cache
import os
import re
import nltk
import joblib
import pyphen
from symspellpy import Verbosity
from g2p_id import G2P as G2P_ID
from g2p_en import G2p as G2P_EN


def _ensure_nltk_resource(resource_path: str, package_name: str):
    try:
        nltk.data.find(resource_path)
    except LookupError:
        nltk.download(package_name, quiet=True)


_ensure_nltk_resource('corpora/cmudict', 'cmudict')
_ensure_nltk_resource('taggers/averaged_perceptron_tagger_eng', 'averaged_perceptron_tagger_eng')
_ensure_nltk_resource('taggers/averaged_perceptron_tagger', 'averaged_perceptron_tagger')

g2p_id_model = G2P_ID()
g2p_en_model = G2P_EN()

try:
    pyphen_en_dict = pyphen.Pyphen(lang='en_US')
except Exception:
    pyphen_en_dict = None

ALPHABET_EJA_MAP = {
    "a": "a", "b": "be", "c": "ce", "d": "de", "e": "e", "f": "ef", "g": "ge",
    "h": "ha", "i": "i", "j": "je", "k": "ka", "l": "el", "m": "em", "n": "en",
    "o": "o", "p": "pe", "q": "ki", "r": "er", "s": "es", "t": "te", "u": "u",
    "v": "ve", "w": "we", "x": "eks", "y": "ye", "z": "zet"
}

# Konversi simbol fonem Inggris (ARPAbet) ke fonetik internasional (IPA)
ARPABET_TO_IPA = {
    "AA": "ɑ", "AE": "æ", "AH": "ʌ", "AO": "ɔ", "AW": "aʊ", "AY": "aɪ",
    "B": "b", "CH": "tʃ", "D": "d", "DH": "ð", "EH": "ɛ", "ER": "ɝ",
    "EY": "eɪ", "F": "f", "G": "ɡ", "HH": "h", "IH": "ɪ", "IY": "i",
    "JH": "dʒ", "K": "k", "L": "l", "M": "m", "NG": "ŋ", "OW": "oʊ",
    "OY": "ɔɪ", "P": "p", "R": "r", "S": "s", "SH": "ʃ", "T": "t",
    "TH": "θ", "UH": "ʊ", "UW": "u", "V": "v", "W": "w", "Y": "j",
    "Z": "z", "ZH": "ʒ"
}




CRF3_VOWELS = {"a", "i", "u", "e", "o", "ə", "ɛ", "ɪ", "ʊ", "ɔ", "æ", "ɐ", "ɒ", "ɑ", "ʌ", "ɝ"}
crf_bi_model = globals().get('crf_bi_model')


def load_crf_bi_model(model_path=None):
    """Load model CRF layer-3 (.joblib)."""
    global crf_bi_model
    target = str(model_path or globals().get('CRF_BI_MODEL_PATH', '')).strip()
    if not target:
        print('[CRF3] Path model kosong. Prediksi BI dinonaktifkan.')
        crf_bi_model = None
        return None
    if not os.path.exists(target):
        print(f'[CRF3] Model tidak ditemukan: {target}. Prediksi BI dinonaktifkan.')
        crf_bi_model = None
        return None

    crf_bi_model = joblib.load(target)
    print(f'[CRF3] Loaded model: {target}')
    return crf_bi_model


CRF3_DIPHTHONGS = ("eɪ", "aɪ", "aʊ", "ɔɪ", "oʊ")


def phoneme_to_units(phoneme: str) -> list[str]:
    text = str(phoneme).strip()
    if not text:
        return []
    if ' ' in text:
        return [u for u in text.split() if u]

    chars = [ch for ch in text if not ch.isspace()]
    units: list[str] = []
    i = 0
    while i < len(chars):
        if i + 1 < len(chars):
            pair = chars[i] + chars[i + 1]
            if pair in CRF3_DIPHTHONGS:
                units.append(pair)
                i += 2
                continue
        units.append(chars[i])
        i += 1
    return units


def _crf3_is_vowel(phone: str) -> bool:
    return phone in CRF3_VOWELS


def _crf3_phone_features(seq: list[str], i: int) -> dict:
    cur = seq[i]
    prev = seq[i - 1] if i > 0 else '<BOS>'
    nxt = seq[i + 1] if i < len(seq) - 1 else '<EOS>'
    return {
        'bias': 1.0,
        'phone': cur,
        'prev_phone': prev,
        'next_phone': nxt,
        'bigram_prev_cur': f'{prev}+{cur}',
        'bigram_cur_next': f'{cur}+{nxt}',
        'is_vowel': _crf3_is_vowel(cur),
        'prev_is_vowel': False if prev == '<BOS>' else _crf3_is_vowel(prev),
        'next_is_vowel': False if nxt == '<EOS>' else _crf3_is_vowel(nxt),
        'pos': i,
        'rev_pos': len(seq) - 1 - i,
        'len_seq': len(seq),
        'bos': i == 0,
        'eos': i == len(seq) - 1,
    }


def predict_bi_from_phoneme(phoneme: str, model=None):
    mdl = model if model is not None else globals().get('crf_bi_model')
    if mdl is None:
        return None

    units = phoneme_to_units(phoneme)
    if not units:
        return None

    feats = [_crf3_phone_features(units, i) for i in range(len(units))]
    pred = mdl.predict_single(feats)
    return ''.join(pred)

def _count_b_from_bi(bi_pattern: str | None) -> int:
    return sum(1 for ch in (bi_pattern or '') if ch == 'B')


def _pyphen_syllable_count(token: str) -> int | None:
    dic = globals().get('pyphen_en_dict')
    if dic is None:
        return None
    t = normalize_typo_token(token)
    if not t:
        return None
    inserted = dic.inserted(t)
    if not inserted:
        return None
    return inserted.count('-') + 1


def _orthographic_vowel_group_count(token: str) -> int:
    t = normalize_typo_token(token)
    if not t:
        return 0
    return len(re.findall(r'[aiueo]+', t))


def is_localized_asg_token(token: str) -> bool:
    t = normalize_typo_token(token)
    if len(t) < 7:
        return False

    strong_english_markers = ('th', 'sh', 'ch', 'ph', 'ck', 'qu', 'tion', 'sion', 'ing')
    if any(m in t for m in strong_english_markers):
        return False

    vowel_letters = sum(ch in 'aiueo' for ch in t)
    vowel_groups = len(re.findall(r'[aiueo]+', t))
    return vowel_letters >= 4 and vowel_groups >= 3


def build_bi_from_target_syllables(units: list[str], target_syllables: int) -> str | None:
    if not units:
        return None

    n = len(units)
    target = max(1, min(int(target_syllables), n))
    if target == 1:
        return 'B' + ('I' * (n - 1))

    labels = ['I'] * n
    labels[0] = 'B'

    candidate = [i for i in range(1, n) if _is_vowel_unit_for_nucleus(units[i])]
    if len(candidate) < target - 1:
        candidate = sorted(set(candidate + list(range(1, n))))

    if not candidate:
        return 'B' + ('I' * (n - 1))

    k = target - 1
    if k >= len(candidate):
        chosen = candidate
    else:
        chosen = []
        for j in range(k):
            idx = round(j * (len(candidate) - 1) / (k - 1)) if k > 1 else 0
            chosen.append(candidate[idx])
        chosen = sorted(set(chosen))

    for idx in chosen:
        labels[idx] = 'B'

    return ''.join(labels)


def refine_asg_bi_with_pyphen(
    token: str,
    units: list[str],
    bi_pattern: str | None,
    switch_route: str | None = None,
) -> tuple[str | None, str | None]:
    if not units:
        return bi_pattern, None

    pyphen_count = _pyphen_syllable_count(token)
    target_count = pyphen_count

    # Fallback untuk kata serapan/kolokial yang sering undercount di pyphen EN
    # (contoh: vaksin) agar tidak terkunci menjadi 1 suku kata.
    if switch_route == 'ASG_G2P_EN_IPA' and (target_count is None or target_count <= 1):
        vowel_groups = _orthographic_vowel_group_count(token)
        if vowel_groups >= 2:
            target_count = vowel_groups

    if target_count is None:
        return bi_pattern, None

    crf_count = _count_b_from_bi(bi_pattern)
    collapsed = crf_count <= 1 and len(units) >= 4
    strongly_mismatch = crf_count > 0 and abs(crf_count - target_count) >= 2
    mild_underseg_en = (
        switch_route == 'ASG_G2P_EN_IPA'
        and crf_count > 0
        and target_count >= 3
        and crf_count < target_count
        and (target_count - crf_count) == 1
    )

    if collapsed or strongly_mismatch or mild_underseg_en:
        rebuilt = build_bi_from_target_syllables(units, target_count)
        if rebuilt:
            return rebuilt, 'ASG_PYPHEN_BI'

    return bi_pattern, None


def choose_asg_phoneme(token: str) -> dict:
    cleaned = normalize_typo_token(token)
    if is_localized_asg_token(cleaned):
        return {
            'switch_route': 'ASG_G2P_ID_LOCALIZED',
            'switch_input': cleaned,
            'phoneme': g2p_id_phoneme(cleaned),
        }

    return {
        'switch_route': 'ASG_G2P_EN_IPA',
        'switch_input': cleaned,
        'phoneme': g2p_foreign_phoneme(cleaned),
    }


def singkatan_to_ejaan(word: str) -> str:
    """dpr -> de pe er"""
    clean = "".join(ch for ch in word.lower() if ch.isalpha())
    return " ".join(ALPHABET_EJA_MAP.get(ch, ch) for ch in clean)


def arpabet_to_ipa(arpabet_tokens: list[str]) -> str:
    ipa_tokens = []
    for token in arpabet_tokens:
        base = re.sub(r"\d", "", token).upper()
        if not base:
            continue
        ipa_tokens.append(ARPABET_TO_IPA.get(base, base.lower()))
    return "".join(ipa_tokens)


def _coerce_g2p_output_to_str(raw) -> str:
    if isinstance(raw, tuple):
        return _coerce_g2p_output_to_str(raw[0])
    if isinstance(raw, list):
        if raw and isinstance(raw[0], list):
            raw = [ph for sub in raw for ph in sub if ph]
        return " ".join(str(part) for part in raw if part)
    return str(raw).strip()


@lru_cache(maxsize=8192)
def g2p_id_phoneme(word: str) -> str:
    if not word:
        return ""
    try:
        res = g2p_id_model.to_phoneme(word)
    except AttributeError:
        try:
            res = g2p_id_model(word)
        except Exception:
            res = g2p_id_model.convert(word)
    return _coerce_g2p_output_to_str(res).replace(" ", "")


@lru_cache(maxsize=8192)
def g2p_foreign_phoneme(word: str) -> str:
    if not word:
        return ""
    raw = g2p_en_model(word)
    phones = [p for p in raw if p and p != " " and p != "<unk>"]
    ipa = arpabet_to_ipa(phones)
    return ipa if ipa else g2p_id_phoneme(word)


def rumus_a_singkatan_simple(word: str) -> str:
    """Rumus A: singkatan -> ejaan alfabet -> fonem Indonesia (g2p-id)."""
    ejaan = singkatan_to_ejaan(word)
    if not ejaan:
        return ""
    return g2p_id_phoneme(ejaan)


# -------------------------------
# Slang lexicon + typo handling (tanpa hardcoded kata)
# -------------------------------

VOWELS = set('aiueo')


def normalize_typo_token(token: str) -> str:
    """Normalisasi typo berbasis bentuk: lowercase, alfabet-only, kompres pengulangan huruf."""
    t = token.lower().strip()
    if not t:
        return t
    t = "".join(ch for ch in t if ch.isalpha())
    if not t:
        return t
    t = re.sub(r"(.)\1{2,}", r"\1\1", t)
    return t


@lru_cache(maxsize=8192)
def symspell_recover_typo(
    token: str,
    allow_no_vowel: bool = False,
    max_edit_distance: int = 2,
    max_len_delta: int = 3,
) -> dict | None:
    """Recover typo ringan; return None jika tidak yakin."""
    model = globals().get('symspell_model')
    if model is None:
        return None

    normalized = normalize_typo_token(token)
    if not normalized:
        return None
    if len(normalized) < 2:
        return None

    if (not allow_no_vowel) and all(ch not in VOWELS for ch in normalized):
        return None

    suggestions = model.lookup(
        normalized,
        Verbosity.TOP,
        max_edit_distance=max_edit_distance,
        include_unknown=True,
        transfer_casing=False,
    )
    if not suggestions:
        return None

    top = suggestions[0]
    recovered = top.term.lower().strip()
    if not recovered or recovered == normalized:
        return None

    allowed_dist = 1 if len(normalized) <= 5 else 2
    allowed_dist = min(allowed_dist, max_edit_distance)
    if top.distance > allowed_dist:
        return None
    if abs(len(recovered) - len(normalized)) > max_len_delta:
        return None

    return {
        'original': normalized,
        'recovered': recovered,
        'distance': int(top.distance),
        'count': int(top.count),
    }


def recover_formal_from_slang(token: str) -> dict | None:
    """Recover formal via map slang.csv dengan exact match setelah normalisasi."""
    mapping = globals().get('slang_formal_map') or {}
    if not mapping:
        return None

    normalized = _norm_lexicon_term(token)
    if not normalized:
        return None

    formal = mapping.get(normalized)
    if not formal:
        return None

    return {
        'original': normalized,
        'matched_slang': normalized,
        'recovered': formal,
        'source': 'exact',
    }


def _recover_formal_with_symspell(token: str) -> dict | None:
    """Recover typo ke kolom formal slang.csv via SymSpell (tanpa map balik slang)."""
    max_edit_distance = int(globals().get('SYMSPELL_TYP_MAX_EDIT_DISTANCE', 1))
    max_len_delta = int(globals().get('SYMSPELL_TYP_MAX_LEN_DELTA', 1))
    require_same_initial = bool(globals().get('SYMSPELL_TYP_REQUIRE_SAME_INITIAL', True))
    require_same_final = bool(globals().get('SYMSPELL_TYP_REQUIRE_SAME_FINAL', True))

    recovered_sym = symspell_recover_typo(
        token,
        allow_no_vowel=True,
        max_edit_distance=max_edit_distance,
        max_len_delta=max_len_delta,
    )
    if not recovered_sym:
        return None

    normalized = normalize_typo_token(token)
    recovered = recovered_sym['recovered']
    if require_same_initial and normalized and recovered and normalized[0] != recovered[0]:
        return None
    if require_same_final and len(normalized) >= 3 and recovered and normalized[-1] != recovered[-1]:
        return None

    formal_terms = globals().get('formal_terms') or set()
    if formal_terms and recovered not in formal_terms:
        return None
    return {
        'original': normalized,
        'matched_slang': None,
        'recovered': recovered,
        'source': 'symspell',
        'symspell': recovered_sym,
    }


def acoustic_switchboard(token: str, label: str, token_meta: dict | None = None) -> dict:
    """Switch board fonem berbasis label IndoBERT + lexicon slang.csv."""
    norm = label.upper().strip()
    cleaned = normalize_typo_token(token)

    if norm in LABELS_SINGKATAN:
        ejaan = singkatan_to_ejaan(token)
        return {
            "switch_route": "A_SINGKATAN_G2P_ID",
            "switch_input": ejaan,
            "phoneme": rumus_a_singkatan_simple(token),
        }

    if norm in LABELS_ASING:
        return choose_asg_phoneme(token)

    if norm in LABELS_DISEMVOWEL:
        recovered = recover_formal_from_slang(token)
        if recovered:
            return {
                "switch_route": "DSM_SLANG_EXACT_G2P_ID",
                "switch_input": recovered['recovered'],
                "phoneme": g2p_id_phoneme(recovered['recovered']),
                "bc_info": {
                    "rule_used": "SLANG_EXACT",
                    "normalized_token": recovered['recovered'],
                    "recover": recovered,
                },
            }

        return {
            "switch_route": "DSM_G2P_ID_DIRECT",
            "switch_input": cleaned,
            "phoneme": g2p_id_phoneme(cleaned),
            "bc_info": {
                "rule_used": "DIRECT",
                "normalized_token": cleaned,
            },
        }

    if norm in LABELS_TYPO:
        recovered = recover_formal_from_slang(token)
        if recovered:
            return {
                "switch_route": "TYP_SLANG_EXACT_G2P_ID",
                "switch_input": recovered['recovered'],
                "phoneme": g2p_id_phoneme(recovered['recovered']),
                "bc_info": {
                    "rule_used": "SLANG_EXACT",
                    "normalized_token": recovered['recovered'],
                    "recover": recovered,
                },
            }

        recovered_sym = _recover_formal_with_symspell(token)
        if recovered_sym:
            return {
                "switch_route": "TYP_FORMAL_SYMSPELL_G2P_ID",
                "switch_input": recovered_sym['recovered'],
                "phoneme": g2p_id_phoneme(recovered_sym['recovered']),
                "bc_info": {
                    "rule_used": "FORMAL_SYM",
                    "normalized_token": recovered_sym['recovered'],
                    "recover": recovered_sym,
                },
            }

        return {
            "switch_route": "TYP_G2P_ID_DIRECT",
            "switch_input": cleaned,
            "phoneme": g2p_id_phoneme(cleaned),
            "bc_info": {
                "rule_used": "DIRECT",
                "normalized_token": cleaned,
            },
        }

    return {
        "switch_route": "O_G2P_ID",
        "switch_input": cleaned,
        "phoneme": g2p_id_phoneme(cleaned),
    }

def _decode_bi_to_syllables(units: list[str], bi_pattern: str | None) -> dict:
    if not units:
        return {
            'valid': False,
            'labels': [],
            'syllables_units': [],
            'syllables_text': [],
            'error': 'units_empty',
        }

    labels = list((bi_pattern or '').strip())
    if len(labels) != len(units) or any(lb not in {'B', 'I'} for lb in labels):
        syll_text = ''.join(units)
        return {
            'valid': False,
            'labels': labels,
            'syllables_units': [units],
            'syllables_text': [syll_text],
            'error': 'invalid_bi_pattern',
        }

    syllables_units: list[list[str]] = []
    current: list[str] = []
    for i, (u, lb) in enumerate(zip(units, labels)):
        if i == 0 or lb == 'B':
            if current:
                syllables_units.append(current)
            current = [u]
        else:
            current.append(u)
    if current:
        syllables_units.append(current)

    syllables_text = [''.join(parts) for parts in syllables_units]
    return {
        'valid': True,
        'labels': labels,
        'syllables_units': syllables_units,
        'syllables_text': syllables_text,
        'error': None,
    }


def _is_vowel_unit_for_nucleus(unit: str) -> bool:
    if not unit:
        return False
    if _crf3_is_vowel(unit):
        return True
    if unit in {'eɪ', 'aɪ', 'aʊ', 'ɔɪ', 'oʊ'}:
        return True
    return any(ch in CRF3_VOWELS for ch in unit)


def _repair_asg_syllable_nucleus(decoded: dict) -> tuple[dict, bool]:
    # Rule-based guardrail pasca-CRF agar setiap silabel ASG punya nucleus vokal.
    # Ini aturan fonotaktik, bukan parameter yang dioptimasi numerik.
    syllables_units = [list(parts) for parts in decoded.get('syllables_units') or []]
    if len(syllables_units) <= 1:
        return decoded, False

    def has_nucleus(parts: list[str]) -> bool:
        return any(_is_vowel_unit_for_nucleus(u) for u in parts)

    changed = False
    i = 0
    step_guard = 0
    step_limit = max(16, len(syllables_units) * 8)
    while i < len(syllables_units):
        step_guard += 1
        if step_guard > step_limit:
            break
        if has_nucleus(syllables_units[i]):
            i += 1
            continue

        if len(syllables_units) <= 1:
            break

        left_idx = i - 1 if i > 0 else None
        right_idx = i + 1 if i + 1 < len(syllables_units) else None
        fixed = False

        if right_idx is not None and syllables_units[right_idx]:
            donor = syllables_units[right_idx]
            if has_nucleus(donor):
                first_vowel_idx = next((j for j, u in enumerate(donor) if _is_vowel_unit_for_nucleus(u)), None)
                donor_keeps_nucleus = any(_is_vowel_unit_for_nucleus(u) for u in donor[1:])
                if first_vowel_idx == 0 and len(donor) >= 2 and donor_keeps_nucleus:
                    syllables_units[i].append(donor.pop(0))
                    changed = True
                    fixed = True

        if (not fixed) and left_idx is not None and syllables_units[left_idx]:
            donor = syllables_units[left_idx]
            if has_nucleus(donor):
                last_vowel_idx = next((j for j in range(len(donor) - 1, -1, -1) if _is_vowel_unit_for_nucleus(donor[j])), None)
                donor_keeps_nucleus = any(_is_vowel_unit_for_nucleus(u) for u in donor[:-1])
                if last_vowel_idx == len(donor) - 1 and len(donor) >= 2 and donor_keeps_nucleus:
                    syllables_units[i].insert(0, donor.pop())
                    changed = True
                    fixed = True

        if not fixed:
            i += 1
            continue

        i = max(i - 1, 0)

    for j in range(len(syllables_units) - 1):
        left = syllables_units[j]
        right = syllables_units[j + 1]
        if len(left) != 1 or not _is_vowel_unit_for_nucleus(left[0]) or not right:
            continue

        first_vowel_idx = next((k for k, u in enumerate(right) if _is_vowel_unit_for_nucleus(u)), None)
        if first_vowel_idx is None:
            continue

        if first_vowel_idx >= 2:
            left.append(right.pop(0))
            changed = True

    if not changed:
        return decoded, False

    repaired = {
        **decoded,
        'syllables_units': syllables_units,
        'syllables_text': [''.join(parts) for parts in syllables_units],
    }
    return repaired, True


def _repair_syllable_nucleus_merge(decoded: dict) -> tuple[dict, bool]:
    syllables_units = [list(parts) for parts in decoded.get('syllables_units') or []]
    if len(syllables_units) <= 1:
        return decoded, False

    def has_nucleus(parts: list[str]) -> bool:
        return any(_is_vowel_unit_for_nucleus(u) for u in parts)

    changed = False
    i = 0
    step_guard = 0
    step_limit = max(16, len(syllables_units) * 8)
    while i < len(syllables_units):
        step_guard += 1
        if step_guard > step_limit:
            break
        if has_nucleus(syllables_units[i]):
            i += 1
            continue

        if len(syllables_units) <= 1:
            break

        left_idx = i - 1 if i > 0 else None
        right_idx = i + 1 if i + 1 < len(syllables_units) else None
        fixed = False

        if right_idx is not None and syllables_units[right_idx]:
            donor = syllables_units[right_idx]
            if has_nucleus(donor):
                first_vowel_idx = next((j for j, u in enumerate(donor) if _is_vowel_unit_for_nucleus(u)), None)
                donor_keeps_nucleus = any(_is_vowel_unit_for_nucleus(u) for u in donor[1:])
                if first_vowel_idx == 0 and len(donor) >= 2 and donor_keeps_nucleus:
                    syllables_units[i].append(donor.pop(0))
                    changed = True
                    fixed = True

        if (not fixed) and left_idx is not None and syllables_units[left_idx]:
            donor = syllables_units[left_idx]
            if has_nucleus(donor):
                last_vowel_idx = next((j for j in range(len(donor) - 1, -1, -1) if _is_vowel_unit_for_nucleus(donor[j])), None)
                donor_keeps_nucleus = any(_is_vowel_unit_for_nucleus(u) for u in donor[:-1])
                if last_vowel_idx == len(donor) - 1 and len(donor) >= 2 and donor_keeps_nucleus:
                    syllables_units[i].insert(0, donor.pop())
                    changed = True
                    fixed = True

        if fixed:
            i = max(i - 1, 0)
            continue


        target_idx = None
        if right_idx is not None and has_nucleus(syllables_units[right_idx]):
            target_idx = right_idx
        elif left_idx is not None and has_nucleus(syllables_units[left_idx]):
            target_idx = left_idx
        elif right_idx is not None:
            target_idx = right_idx
        elif left_idx is not None:
            target_idx = left_idx

        if target_idx is None:
            i += 1
            continue

        stranded = syllables_units[i]
        if target_idx < i:
            syllables_units[target_idx].extend(stranded)
            syllables_units.pop(i)
            changed = True
            i = max(target_idx, 0)
            continue

        syllables_units[target_idx] = stranded + syllables_units[target_idx]
        syllables_units.pop(i)
        changed = True

    if not changed:
        return decoded, False

    repaired = {
        **decoded,
        'syllables_units': syllables_units,
        'syllables_text': [''.join(parts) for parts in syllables_units],
    }
    return repaired, True


def _repair_id_vowel_hiatus(decoded: dict, token: str, switch_route: str | None) -> tuple[dict, bool]:
    # Rule khusus Indonesia untuk pola hiatus vokal tertentu (mis. ai/au/oi/ei)
    # agar pemenggalan lebih natural pada kasus umum; tidak berasal dari tuning threshold.
    if switch_route != 'O_G2P_ID':
        return decoded, False

    syllables_units = [list(parts) for parts in decoded.get('syllables_units') or []]
    if len(syllables_units) <= 1:
        return decoded, False

    t = normalize_typo_token(token)
    changed = False

    # Merge pola vokal tunggal terpisah (mis. na | i -> nai pada "menunaikan").
    if any(dg in t for dg in ('ai', 'au', 'oi', 'ei')):
        i = 1
        while i < len(syllables_units) - 1:
            prev = syllables_units[i - 1]
            cur = syllables_units[i]
            nxt = syllables_units[i + 1]
            if (
                len(cur) == 1
                and _is_vowel_unit_for_nucleus(cur[0])
                and prev
                and _is_vowel_unit_for_nucleus(prev[-1])
                and nxt
                and (not _is_vowel_unit_for_nucleus(nxt[0]))
            ):
                prev.extend(cur)
                syllables_units.pop(i)
                changed = True
                continue
            i += 1

    # Split pola ...au|han menjadi ...a|u|han (mis. "berjauhan").
    i = 0
    while i < len(syllables_units) - 1:
        left = syllables_units[i]
        right = syllables_units[i + 1]
        if (
            len(left) >= 2
            and _is_vowel_unit_for_nucleus(left[-1])
            and _is_vowel_unit_for_nucleus(left[-2])
            and right
            and right[0] == 'h'
            and len(right) >= 2
            and _is_vowel_unit_for_nucleus(right[1])
            and 'auh' in t
        ):
            trailing_vowel = left.pop()
            syllables_units.insert(i + 1, [trailing_vowel])
            changed = True
            i += 2
            continue
        i += 1

    if not changed:
        return decoded, False

    repaired = {
        **decoded,
        'syllables_units': syllables_units,
        'syllables_text': [''.join(parts) for parts in syllables_units],
    }
    return repaired, True


def _extract_rhyme_tail(syllable_text: str) -> str:
    if not syllable_text:
        return ''
    chars = [ch for ch in syllable_text if not ch.isspace()]
    if not chars:
        return ''
    for i in range(len(chars) - 1, -1, -1):
        if chars[i] in CRF3_VOWELS:
            return ''.join(chars[i:])
    return ''.join(chars)


def analyze_line_prosody_full(text: str) -> dict:
    token_rows = predict_with_routing(text)
    postprocess_enabled = bool(globals().get('ENABLE_POST_PROCESSING', True))
    analyzed_tokens = []
    line_syllables = []
    line_units = []
    line_bi_flat = []

    def _has_nucleus(parts: list[str]) -> bool:
        return any(_is_vowel_unit_for_nucleus(u) for u in parts)

    for row in token_rows:
        token_label = row.get('label')
        if token_label in LABELS_SINGKATAN:
            token_syllables = [u for u in singkatan_to_ejaan(row.get('token') or '').split() if u]
            line_syllables.extend(token_syllables)
            analyzed_tokens.append({
                **row,
                'units': [],
                'unit_count': 0,
                'bi_pattern': '',
                'bi_valid': True,
                'bi_error': None,
                'syllables': token_syllables,
                'syllable_count': len(token_syllables),
                'rhyme_tail_token': _extract_rhyme_tail(token_syllables[-1]) if token_syllables else '',
                'asg_nucleus_fixed': False,
                'syllable_repair': 'SKT_SKIP_POSTPROCESS',
            })
            continue

        phoneme = row.get('phoneme') or ''
        units = phoneme_to_units(phoneme)
        bi_pattern = row.get('bi_pattern')
        if not bi_pattern:
            bi_pattern = predict_bi_from_phoneme(phoneme)

        decoded = _decode_bi_to_syllables(units, bi_pattern)
        asg_nucleus_fixed = False
        syllable_repair = None
        if postprocess_enabled:
            if str(row.get('switch_route', '')).startswith('ASG_'):
                decoded, asg_nucleus_fixed = _repair_asg_syllable_nucleus(decoded)
                if asg_nucleus_fixed:
                    syllable_repair = 'asg_nucleus_fix'
                has_stranded = any((not _has_nucleus(parts)) for parts in (decoded.get('syllables_units') or []))
                if has_stranded:
                    decoded, merged_after_asg = _repair_syllable_nucleus_merge(decoded)
                    if merged_after_asg:
                        syllable_repair = 'asg_nucleus_merge'
            else:
                decoded, generic_nucleus_fixed = _repair_syllable_nucleus_merge(decoded)
                if generic_nucleus_fixed:
                    syllable_repair = 'nucleus_merge'
                decoded, id_hiatus_fixed = _repair_id_vowel_hiatus(
                    decoded,
                    token=row.get('token') or '',
                    switch_route=row.get('switch_route'),
                )
                if id_hiatus_fixed:
                    syllable_repair = 'id_vowel_hiatus_fix'
        else:
            syllable_repair = 'POSTPROCESS_DISABLED'
        token_syllables = decoded['syllables_text']

        line_syllables.extend(token_syllables)
        line_units.extend(units)
        if decoded['valid']:
            line_bi_flat.extend(decoded['labels'])

        analyzed = {
            **row,
            'units': units,
            'unit_count': len(units),
            'bi_pattern': bi_pattern,
            'bi_valid': decoded['valid'],
            'bi_error': decoded['error'],
            'syllables': token_syllables,
            'syllable_count': len(token_syllables),
            'rhyme_tail_token': _extract_rhyme_tail(token_syllables[-1]) if token_syllables else '',
            'asg_nucleus_fixed': asg_nucleus_fixed,
            'syllable_repair': syllable_repair,
        }
        analyzed_tokens.append(analyzed)

    total_syllables = len(line_syllables)
    final_syllable = line_syllables[-1] if line_syllables else ''
    final_rhyme = _extract_rhyme_tail(final_syllable)

    return {
        'input_text': text,
        'tokens': analyzed_tokens,
        'line_units': line_units,
        'line_unit_count': len(line_units),
        'line_bi_pattern_flat': ''.join(line_bi_flat),
        'line_syllables': line_syllables,
        'line_syllable_count': total_syllables,
        'final_syllable': final_syllable,
        'final_rhyme': final_rhyme,
    }


def print_line_prosody_analysis(report: dict) -> None:
    print('=== ANALISIS LENGKAP LAYER-1 -> LAYER-3 ===')
    print(f"Input text         : {report['input_text']}")
    print(f"Total token        : {len(report['tokens'])}")
    print()

    print('--- Detail per token ---')
    for idx, row in enumerate(report['tokens'], start=1):
        conf = row.get('model_confidence')
        margin = row.get('model_margin')
        conf_txt = f"{conf:.3f}" if conf is not None else '-'
        margin_txt = f"{margin:.3f}" if margin is not None else '-'

        print(
            f"[{idx:02d}] token={row['token']} | raw={row['raw_label']} | final={row['label']} | "
            f"route={row['route']} | switch={row['switch_route']} | conf={conf_txt} margin={margin_txt}"
        )
        if row.get('bc_info'):
            bc = row['bc_info']
            print(f"     bc_rule={bc.get('rule_used')} | normalized={bc.get('normalized_token')}")
        print(f"     phoneme={row.get('phoneme', '')}")
        print(f"     units={row['units']} (n={row['unit_count']})")
        print(
            f"     bi={row.get('bi_pattern') or '-'} | bi_valid={row.get('bi_valid')}"
            + (f" | bi_error={row.get('bi_error')}" if row.get('bi_error') else '')
        )
        print(f"     syllables={row.get('syllables', [])} | syllable_count={row.get('syllable_count', 0)}")
        if row.get('syllable_repair'):
            print(f"     postprocess={row.get('syllable_repair')}")
        print(f"     token_rhyme_tail={row.get('rhyme_tail_token', '')}")

    print('\n--- Agregasi baris ---')
    print(f"line_units          : {report['line_units']}")
    print(f"line_unit_count     : {report['line_unit_count']}")
    print(f"line_bi_pattern     : {report['line_bi_pattern_flat']}")
    print(f"line_syllables      : {report['line_syllables']}")
    print(f"JUMLAH_SILABEL      : {report['line_syllable_count']}")
    print(f"SILABEL_AKHIR       : {report['final_syllable']}")
    print(f"RIMA_AKHIR          : {report['final_rhyme']}")

def _rhyme_key(rhyme_tail: str) -> str:
    t = (rhyme_tail or '').strip()
    if not t:
        return '-'
    if len(t) >= 2:
        return t[-2:]
    return t


def infer_rhyme_scheme(line_reports: list[dict]) -> str:
    keys = [_rhyme_key(rep.get('final_rhyme', '')) for rep in line_reports]
    mapping = {}
    next_label_ord = ord('a')
    scheme = []
    for key in keys:
        if key not in mapping:
            mapping[key] = chr(next_label_ord)
            next_label_ord += 1
        scheme.append(mapping[key])
    return '-'.join(scheme)


def analyze_pantun_multiline(text: str) -> dict:
    lines = [ln.strip() for ln in str(text).splitlines() if ln.strip()]
    line_reports = []
    for i, line in enumerate(lines, start=1):
        rep = analyze_line_prosody_full(line)
        rep['line_no'] = i
        line_reports.append(rep)

    rhyme_scheme = infer_rhyme_scheme(line_reports)
    total_syllables = sum(rep.get('line_syllable_count', 0) for rep in line_reports)

    return {
        'raw_text': text,
        'line_count': len(line_reports),
        'lines': line_reports,
        'total_syllables_all_lines': total_syllables,
        'rhyme_scheme': rhyme_scheme,
    }


def build_pantun_compact_output(report: dict) -> dict:
    lines = report.get('lines', [])

    compact = {
        'skema_rima': report.get('rhyme_scheme', ''),
        'rima_akhir_baris_1': '',
        'rima_akhir_baris_2': '',
        'rima_akhir_baris_3': '',
        'rima_akhir_baris_4': '',
        'suku_kata_baris_1': '',
        'suku_kata_baris_2': '',
        'suku_kata_baris_3': '',
        'suku_kata_baris_4': '',
    }

    for i in range(min(4, len(lines))):
        rep = lines[i]
        compact[f'rima_akhir_baris_{i+1}'] = rep.get('final_rhyme', '')
        compact[f'suku_kata_baris_{i+1}'] = rep.get('line_syllable_count', '')

    return compact


def print_pantun_multiline_analysis(report: dict, verbose: bool = False) -> None:
    compact = build_pantun_compact_output(report)
    cols = [
        'skema_rima',
        'rima_akhir_baris_1',
        'rima_akhir_baris_2',
        'rima_akhir_baris_3',
        'rima_akhir_baris_4',
        'suku_kata_baris_1',
        'suku_kata_baris_2',
        'suku_kata_baris_3',
        'suku_kata_baris_4',
    ]

    print(','.join(str(compact[col]) for col in cols))

    if not verbose:
        return

    print('\n=== ANALISIS PANTUN MULTI-BARIS (VERBOSE) ===')
    print(f"Jumlah baris terdeteksi : {report['line_count']}")
    print(f"Total silabel (semua)   : {report['total_syllables_all_lines']}")
    print(f"Skema rima              : {report['rhyme_scheme']}")

    for line_rep in report['lines']:
        print('\n' + '=' * 64)
        print(f"BARIS {line_rep['line_no']}: {line_rep['input_text']}")
        print_line_prosody_analysis(line_rep)





In [ ]:
# Contoh uji input pantun multi-baris (pisahkan dengan newline "\n")
sample_pantun = """lagi ngeliat ilc sekarang\npetinggi sunda empire bikin ngakak\ncintaku jangan kau buang\naku gamau jantungku berhenti mendadak"""

pantun_report = analyze_pantun_multiline(sample_pantun)

# Default: output ringkas sesuai format dataset
print_pantun_multiline_analysis(pantun_report)

# Verbose (opsional): tampilkan semua detail per token/baris
# print_pantun_multiline_analysis(pantun_report, verbose=True)
